In [1]:
import pandas as pd
order = pd.read_csv('../data/raw/orders_raw.csv')
product = pd.read_csv('../data/raw/products_raw.csv')



In [2]:
product.isnull().sum()
product["tags"] = product["tags"].fillna("No tags")

product["product_type"] = product["product_type"].fillna("snowboard")
product.isnull().sum()

product_id       0
product_title    0
product_type     0
tags             0
variant_id       0
sku              0
current_stock    0
price            0
dtype: int64

In [3]:
order["order_date"] = pd.to_datetime(order["order_date"])


In [4]:
order

,order_id,order_date,financial_status,product_id,variant_id,sku,product_title,quantity,unit_price,total_price
0,10000,2022-01-01,paid,26733,6486,LDS-001,Linen Duvet Set,5,89.99,449.95
1,10001,2022-01-01,paid,39309,87418,CVB-002,Ceramic Vase Bundle,2,45.00,90.00
2,10002,2022-01-01,paid,94499,81074,SCP-003,Scented Candle Pack,6,29.99,179.94
3,10003,2022-01-01,paid,27964,27855,WAB-004,Wall Art Boho,3,65.00,195.00
4,10004,2022-01-01,paid,49372,3871,TRS-005,Table Runner Set,1,22.50,22.50
...,...,...,...,...,...,...,...,...,...,...
6501,16501,2025-01-01,paid,39309,87418,CVB-002,Ceramic Vase Bundle,3,45.00,135.00
6502,16502,2025-01-01,paid,94499,81074,SCP-003,Scented Candle Pack,4,29.99,119.96
6503,16503,2025-01-01,paid,27964,27855,WAB-004,Wall Art Boho,2,65.00,130.00
6504,16504,2025-01-01,paid,49372,3871,TRS-005,Table Runner Set,2,22.50,45.00


In [5]:
df = order.copy()

daily = df.groupby(["sku","product_title","order_date"]).agg(
units_sold = ("quantity", "sum"),
revenue = ("total_price", "sum"),
num_orders = ("order_id", "nunique"),
avg_price = ("unit_price", "mean")
).reset_index()
all_skus = daily["sku"].unique()
date_range = pd.date_range(
            start = daily["order_date"].min(),
            end   = daily["order_date"].max(),
            freq  = "D"
        )
idx = pd.MultiIndex.from_product([all_skus,date_range], names=["sku", "order_date" ])
daily = daily.set_index(["sku", "order_date"]).reindex(idx, fill_value=0).reset_index()
title_map = daily.groupby("sku")["product_title"].first()
daily["product_title"] = daily["sku"].map(title_map)
df = daily

In [6]:

df = df.sort_values(["sku", "order_date"])

        # ── DATE FEATURES ──────────────────────────────────
df["day_of_week"]   = df["order_date"].dt.dayofweek    # 0=Mon 6=Sun
df["day_of_month"]  = df["order_date"].dt.day
df["week_of_year"]  = df["order_date"].dt.isocalendar().week.astype(int)
df["month"]         = df["order_date"].dt.month
df["quarter"]       = df["order_date"].dt.quarter
df["year"]          = df["order_date"].dt.year
df["is_weekend"]    = df["day_of_week"].isin([4, 5, 6]).astype(int)
# In Pakistan, Friday is the big day (Jumma)
df["is_friday"]     = (df["day_of_week"] == 4).astype(int)
# Month-end salary effect (25th-31st)
df["is_month_end"]  = (df["day_of_month"] >= 25).astype(int)

# ── PAKISTANI SEASONAL FEATURES ────────────────────
# These are the most important features for NestHaven

# Nov-Dec holiday season (peak)
df["is_holiday_season"] = df["month"].isin([11, 12]).astype(int)

# Eid ul-Fitr — moves each year (lunar calendar)
# Approximate Gregorian dates for 2022-2025
eid_fitr_dates = [
    "2022-05-02", "2022-05-03",
    "2023-04-21", "2023-04-22",
    "2024-04-10", "2024-04-11",
    "2025-03-30", "2025-03-31",
]
# 14-day pre-Eid window (shopping peaks before Eid)
eid_fitr_windows = []
for d in eid_fitr_dates:
    dt = pd.to_datetime(d)
    for i in range(-14, 3):
        eid_fitr_windows.append(dt + pd.Timedelta(days=i))

        # Eid ul-Adha — approximate dates
eid_adha_dates = [
    "2022-07-09", "2022-07-10",
    "2023-06-28", "2023-06-29",
    "2024-06-17", "2024-06-18",
    "2025-06-06", "2025-06-07",
]
eid_adha_windows = []
for d in eid_adha_dates:
    dt = pd.to_datetime(d)
    for i in range(-14, 3):
        eid_adha_windows.append(dt + pd.Timedelta(days=i))

df["is_eid_fitr_window"] = df["order_date"].isin(eid_fitr_windows).astype(int)
df["is_eid_adha_window"] = df["order_date"].isin(eid_adha_windows).astype(int)
df["is_eid_season"]      = ((df["is_eid_fitr_window"] == 1) | (df["is_eid_adha_window"] == 1)).astype(int)

# Combined peak flag — any major event
df["is_peak_season"] = ((df["is_holiday_season"] == 1) | (df["is_eid_season"] == 1)).astype(int)

# ── LAG FEATURES ───────────────────────────────────
# "What did this SKU sell yesterday / last week?"
# XGBoost needs these to understand momentum
for lag in [1, 3, 7, 14, 21, 30]:
    df[f"lag_{lag}d"] = df.groupby("sku")["units_sold"].shift(lag)

# ── ROLLING WINDOW FEATURES ────────────────────────
# Smoothed averages — removes daily noise
for window in [7, 14, 30, 60, 90]:
    df[f"rolling_mean_{window}d"] = (
        df.groupby("sku")["units_sold"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f"rolling_std_{window}d"] = (
        df.groupby("sku")["units_sold"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std().fillna(0))
    )

        # ── TREND FEATURES ─────────────────────────────────
        # Is this SKU growing or declining?
df["trend_7_vs_30"] = (
    df["rolling_mean_7d"] / (df["rolling_mean_30d"] + 1e-5)
)  # >1 means accelerating, <1 means slowing

        # ── CUMULATIVE FEATURES ────────────────────────────
df["cumulative_units"] = df.groupby("sku")["units_sold"].cumsum()

        # ── PRODUCT FEATURES ───────────────────────────────
        # Merge product metadata

product_meta = product[["sku", "product_type", "price", "current_stock"]].copy()
product_meta["sku"] = product_meta["sku"].str.upper().str.strip()
df = df.merge(product_meta, on="sku", how="left")

        # Encode product type as category code
df["product_type_code"] = df["product_type"].astype("category").cat.codes

        # Price tier — budget / mid / premium
df["price_tier"] = pd.cut(
    df["price"],
    bins   = [0, 25, 60, float("inf")],
    labels = [0, 1, 2]  # 0=budget, 1=mid, 2=premium
).astype(float)

        # Drop rows with NaN lags (first few days per SKU have no lag data)
df = df.dropna(subset=["lag_7d", "rolling_mean_7d"])


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6540 entries, 7 to 6581
Data columns (total 44 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   sku                 6540 non-null   object        
 1   order_date          6540 non-null   datetime64[ns]
 2   product_title       6540 non-null   object        
 3   units_sold          6540 non-null   int64         
 4   revenue             6540 non-null   float64       
 5   num_orders          6540 non-null   int64         
 6   avg_price           6540 non-null   float64       
 7   day_of_week         6540 non-null   int32         
 8   day_of_month        6540 non-null   int32         
 9   week_of_year        6540 non-null   int64         
 10  month               6540 non-null   int32         
 11  quarter             6540 non-null   int32         
 12  year                6540 non-null   int32         
 13  is_weekend          6540 non-null   int64         
 1

In [9]:
# setting data for ml model
import pandas as pd
df = pd.read_csv("../data/features/features.csv")
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.sort_values(by='order_date').reset_index(drop=True)
df = df.drop(columns=["sku","order_date","product_title","product_type"]) # product type because thier only one product snowboard
lag_cols = ['lag_1d', 'lag_3d', 'lag_7d', 'lag_14d', 'lag_21d', 'lag_30d']
df = df.dropna(subset=lag_cols)



In [10]:
# ML model global to predict units sold

# Define split percentage (e.g., 80% train, 20% test)
train_size = int(len(df) * 0.8)

# Split the data chronologically
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]
X_train = train_df.drop(columns=["units_sold"])
y_train = train_df["units_sold"]
X_test = test_df.drop(columns=["units_sold"])
y_test = test_df["units_sold"]


In [11]:
import xgboost as xgb
model = xgb.XGBRegressor(
    n_estimators = 500,
    max_depth = 6,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,


)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error,root_mean_squared_error
mape = mean_absolute_percentage_error(y_pred,y_test)
mae = mean_absolute_error(y_pred,y_test)
rmse = root_mean_squared_error(y_pred,y_test)
print(f"rmse = {rmse} , mae= {mae}, mape= {mape}")

rmse = 0.9195877313613892 , mae= 0.308400958776474, mape= 0.02764766849577427


In [12]:
df = pd.read_csv("../data/features/features.csv")

# Check if Eid season actually has higher sales
print("Average daily sales by season:")
print(df.groupby("is_eid_season")["units_sold"].mean())
print(df.groupby("is_holiday_season")["units_sold"].mean())

# Check daily variance
print("\nSales std per SKU:")
print(df.groupby("sku")["units_sold"].std().sort_values(ascending=False))

Average daily sales by season:
is_eid_season
0     8.762050
1    19.896605
Name: units_sold, dtype: float64
is_holiday_season
0     8.624587
1    16.014572
Name: units_sold, dtype: float64

Sales std per SKU:
sku
SCP-003    12.317829
LDS-001     8.717284
TPC-006     7.626233
CVB-002     5.737241
WAB-004     4.695259
TRS-005     2.426515
Name: units_sold, dtype: float64


In [ ]:
import pickle
from datetime import datetime, timedelta
import pandas as pd
def prediction(self, forecast_days: int = 30):
    with open('models/xgboost_model.pkl', 'rb') as f:
        model = pickle.load(f)

    # Explicitly list your model's 44 training features in their exact training order:
    MODEL_FEATURES = [
        "day_of_week", "day_of_month", "week_of_year", "month", "quarter", "year",
        "is_weekend", "is_friday", "is_month_end", "is_holiday_season",
        "is_eid_fitr_window", "is_eid_adha_window", "is_eid_season",
        "is_peak_season", "lag_1d", "lag_3d", "lag_7d", "lag_14d",
        "lag_21d", "lag_30d", "rolling_mean_7d", "rolling_std_7d",
        "rolling_mean_14d", "rolling_std_14d", "rolling_mean_30d",
        "rolling_std_30d", "rolling_mean_60d", "rolling_std_60d",
        "rolling_mean_90d", "rolling_std_90d", "sku_mean_demand",
        "sku_std_demand", "sku_demand_cv", "trend_7_vs_30",
        "cumulative_units", "sku_x_eid", "sku_x_holiday", "sku_x_peak",
        "sku_x_weekend", "demand_x_lag", "price", "current_stock",
        "product_type_code", "price_tier"
    ]

    # Load your current stock snapshot
    input_df = pd.read_csv('data/raw/products_raw.csv')

    # Encode product categories to match training mappings
    type_mapping = {
        "Bedding": 0,
        "Decor": 1,
        "Fragrance": 2,
        "Kitchen": 3
    }

    input_df['product_type_code'] = (
        input_df['product_type']
        .map(type_mapping)
        .fillna(0)
        .astype(int)
    )

    # Create a simplistic rule for your price_tier feature
    input_df['price_tier'] = (
        pd.qcut(
            input_df['price'],
            q=3,
            labels=[1, 2, 3],
            duplicates='drop'
        ).astype(float)
    )

    # =====================================================================
    # 2. SEED HISTORICAL SALES (CRITICAL FOR LAGS)
    # =====================================================================
    history_records = []
    start_date = datetime.now()

    for _, row in input_df.iterrows():
        base_demand = 2.0 if "high" in str(row['tags']) else 0.5

        for day_offset in range(95, 0, -1):
            hist_date = start_date - timedelta(days=day_offset)

            history_records.append({
                'sku': row['sku'],
                'order_date': hist_date.date(),
                'units_sold': max(
                    0.0,
                    np.random.normal(base_demand, base_demand * 0.2)
                ),
                'price': row['price'],
                'current_stock': row['current_stock'],
                'product_type_code': row['product_type_code'],
                'price_tier': row['price_tier']
            })

    history_df = pd.DataFrame(history_records)

    # =====================================================================
    # 3. RECURSIVE FORECAST LOOP
    # =====================================================================
    forecast_results = []

    for day in range(1, forecast_days + 1):
        target_date = start_date + timedelta(days=day)

        day_features = {
            'order_date': target_date.date(),
            'day_of_week': target_date.weekday(),
            'day_of_month': target_date.day,
            'week_of_year': target_date.isocalendar()[1],
            'month': target_date.month,
            'quarter': (target_date.month - 1) // 3 + 1,
            'year': target_date.year,
            'is_weekend': 1 if target_date.weekday() >= 5 else 0,
            'is_friday': 1 if target_date.weekday() == 4 else 0,
            'is_month_end': 1 if (target_date + timedelta(days=1)).day == 1 else 0,
            'is_holiday_season': 1 if target_date.month in [11, 12] else 0,
            'is_eid_fitr_window': 0,
            'is_eid_adha_window': 0,
            'is_eid_season': 0,
            'is_peak_season': 0
        }

        for _, sku_row in input_df.iterrows():
            sku = sku_row['sku']

            sku_hist = (
                history_df[history_df['sku'] == sku]
                .sort_values('order_date')
            )

            past_sales = sku_hist['units_sold'].values

            lags = {
                'lag_1d': past_sales[-1] if len(past_sales) >= 1 else 0,
                'lag_3d': past_sales[-3] if len(past_sales) >= 3 else 0,
                'lag_7d': past_sales[-7] if len(past_sales) >= 7 else 0,
                'lag_14d': past_sales[-14] if len(past_sales) >= 14 else 0,
                'lag_21d': past_sales[-21] if len(past_sales) >= 21 else 0,
                'lag_30d': past_sales[-30] if len(past_sales) >= 30 else 0,
            }

            rolling = {
                'rolling_mean_7d': np.mean(past_sales[-7:]),
                'rolling_std_7d': np.std(past_sales[-7:]),
                'rolling_mean_14d': np.mean(past_sales[-14:]),
                'rolling_std_14d': np.std(past_sales[-14:]),
                'rolling_mean_30d': np.mean(past_sales[-30:]),
                'rolling_std_30d': np.std(past_sales[-30:]),
                'rolling_mean_60d': np.mean(past_sales[-60:]),
                'rolling_std_60d': np.std(past_sales[-60:]),
                'rolling_mean_90d': np.mean(past_sales[-90:]),
                'rolling_std_90d': np.std(past_sales[-90:])
            }

            profile = {
                'sku_mean_demand': np.mean(past_sales),
                'sku_std_demand': np.std(past_sales),
                'sku_demand_cv': (
                    np.std(past_sales) /
                    (np.mean(past_sales) + 1e-5)
                ),
                'trend_7_vs_30': (
                    np.mean(past_sales[-7:]) /
                    (np.mean(past_sales[-30:]) + 1e-5)
                ),
                'cumulative_units': int(np.sum(past_sales))
            }

            interactions = {
                'sku_x_eid':
                    profile['sku_mean_demand'] * day_features['is_eid_season'],
                'sku_x_holiday':
                    profile['sku_mean_demand'] * day_features['is_holiday_season'],
                'sku_x_peak':
                    profile['sku_mean_demand'] * day_features['is_peak_season'],
                'sku_x_weekend':
                    profile['sku_mean_demand'] * day_features['is_weekend'],
                'demand_x_lag':
                    profile['sku_mean_demand'] * lags['lag_1d']
            }

            full_row = {
                **day_features,
                **lags,
                **rolling,
                **profile,
                **interactions,
                'sku': sku,
                'price': sku_row['price'],
                'current_stock': sku_row['current_stock'],
                'product_type_code': sku_row['product_type_code'],
                'price_tier': sku_row['price_tier']
            }

            X_test = pd.DataFrame([full_row])[MODEL_FEATURES]

            predicted_units = model.predict(X_test)[0]
            predicted_units = max(0, predicted_units)

            new_history_row = {
                'sku': sku,
                'order_date': target_date.date(),
                'units_sold': predicted_units,
                'price': sku_row['price'],
                'current_stock': sku_row['current_stock'],
                'product_type_code': sku_row['product_type_code'],
                'price_tier': sku_row['price_tier']
            }

            history_df = pd.concat(
                [history_df, pd.DataFrame([new_history_row])],
                ignore_index=True
            )

            forecast_results.append({
                'sku': sku,
                'date': target_date.date(),
                'predicted_sales': predicted_units
            })

    # =====================================================================
    # 4. AGGREGATE RESULTS & METRICS
    # =====================================================================
    forecast_output = pd.DataFrame(forecast_results)

    summary = (
        forecast_output
        .groupby('sku')
        .agg(
            total_30d_demand=('predicted_sales', 'sum'),
            avg_daily_demand=('predicted_sales', 'mean')
        )
        .reset_index()
    )

    final_report = summary.merge(
        input_df[['sku', 'product_title', 'current_stock']],
        on='sku'
    )

    final_report['remaining_stock_after_30d'] = (
        final_report['current_stock'] -
        final_report['total_30d_demand']
    )

    final_report['stockout_risk'] = (
        final_report['remaining_stock_after_30d']
        .apply(lambda x: "HIGH RISK" if x <= 0 else "SAFE")
    )

    print(final_report.to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6540 entries, 0 to 6539
Data columns (total 52 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   sku                 6540 non-null   object 
 1   order_date          6540 non-null   object 
 2   product_title       6540 non-null   object 
 3   units_sold          6540 non-null   int64  
 4   revenue             6540 non-null   float64
 5   num_orders          6540 non-null   int64  
 6   avg_price           6540 non-null   float64
 7   day_of_week         6540 non-null   int64  
 8   day_of_month        6540 non-null   int64  
 9   week_of_year        6540 non-null   int64  
 10  month               6540 non-null   int64  
 11  quarter             6540 non-null   int64  
 12  year                6540 non-null   int64  
 13  is_weekend          6540 non-null   int64  
 14  is_friday           6540 non-null   int64  
 15  is_month_end        6540 non-null   int64  
 16  is_hol